In [1]:
%pip install google-generativeai langgraph tavily
%pip install tavily-python
from tavily import TavilyClient

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for tavily: filename=tavily-1.1.0-py3-none-any.whl size=6185 sha256=9258f8d8429281947b8341c6f806419a8894e1f992d82d83e026987088ed39e7
  Stored in directory: /Users/stell/Library/Caches/pip/wheels/51/08/fd/a42b1feea25355e9e6357061510b277422f42ee1f044aa24d3
Successfully built tavily

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached tiktoken-0.12.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (6.7 kB)
Using cached tiktoken-0.12.0-cp314-cp314-macosx_11_0_arm64.whl (993 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [tavily-python]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated pac

In [2]:
import os 
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv()


os.environ['GOOGLE_API_KEY'] = os.getenv('GEMINI_API_KEY') 
os.environ['TAVILY_API_KEY'] = os.getenv('TAVILY_API_KEY')


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/l2/t02s0r_51x727q8t6t5_qbp40000gn/T/ipykernel_69892/3266602130.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
client = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

result = client.search("O que são multiagentes de Inteligência Artificial?",
                       include_answer=True)

result["answer"]

'Multiagentes de Inteligência Artificial são sistemas compostos por vários agentes autônomos que trabalham juntos para resolver problemas. Eles interagem e coordenam suas ações para alcançar objetivos comuns. A inteligência em multiagentes surge da colaboração entre os agentes.'

In [4]:
%pip install -U duckduckgo_search

  Using cached lxml-6.0.2-cp314-cp314-macosx_10_13_universal2.whl.metadata (3.6 kB)
Using cached lxml-6.0.2-cp314-cp314-macosx_10_13_universal2.whl (8.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 5.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [duckduckgo_search]lxml]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
cidade = "Belém"

query = f"""
Liste os 5 principais restaurantes em {cidade}, segundo avaliações recentes no TripAdvisor ou sites similares de turismo.
Para cada restaurante, informe:
- Tipo de culinária (ex: regional, italiana, japonesa)
- Uma breve descrição (máx. 2 linhas)
- Avaliação média (se disponível)
- Faixa de preço

Responda apenas com dados atualizados e relevantes para turistas.
"""

In [13]:
from duckduckgo_search import DDGS
import re

ddg = DDGS()
def search(query, max_results=6):
    try: 
        results = ddg.text(query, max_results=max_results)
        return [i["href"] for i in results]
    except Exception as e:
        raise e
for link in search(query):
    print(link)

/var/folders/l2/t02s0r_51x727q8t6t5_qbp40000gn/T/ipykernel_69892/2188133513.py:4: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddg = DDGS()


In [14]:
import os
import re
from tavily import TavilyClient

api_key = os.environ.get("TAVILY_API_KEY")
if not api_key:
    raise ValueError("A chave TAVILY_API_KEY não foi encontrada.")

cliente_tavily = TavilyClient(api_key=api_key)

cidade = "Belém do Pará"
tavily_query = f"restaurantes em {cidade} tripadvisor com maior quantidade de reviews e faixa de preço"


print("Iniciando busca agêntica por URLs do Tripadvisor com Tavily...")
tripadvisor_url = None
try:
    tavily_results = cliente_tavily.search(query=tavily_query, max_results=5)

    if tavily_results and tavily_results["results"]:
        print(f"Tavily encontrou {len(tavily_results['results'])} resultados. Analisando...")

        for result in tavily_results["results"]:
            url = result["url"]
            
            if "tripadvisor.com" in url or "tripadvisor.com.br" in url:
                tripadvisor_url = url
                break

        if not tripadvisor_url:
            print("Nenhum URL relevante do Tripadvisor foi encontrado nos primeiros resultados.")
            
    else:
        print("Tavily não encontrou resultados para a busca agêntica.")

except Exception as e:
    print(f"Erro na busca agêntica com Tavily: {e}. Verifique sua chave API ou conexão.")

if tripadvisor_url:
    clean_url = re.sub(r'-oa\d+-', '-', tripadvisor_url)
    tripadvisor_url = clean_url
    print(f"  ✅ URL encontrada limpa de paginação.")

print("-" * 50)
print(f"URL Final do Tripadvisor para raspagem: {tripadvisor_url if tripadvisor_url else 'NÃO ENCONTRADO'}")
print("-" * 50)

Iniciando busca agêntica por URLs do Tripadvisor com Tavily...
Tavily encontrou 5 resultados. Analisando...
  ✅ URL encontrada limpa de paginação.
--------------------------------------------------
URL Final do Tripadvisor para raspagem: https://www.tripadvisor.com.br/Restaurants-g303404-c22-Belem_State_of_Para.html
--------------------------------------------------


In [ ]:
from bs4 import BeautifulSoup
import re # Importar regex para extração mais limpa

# Este é o HTML completo que você me forneceu - Como se estivessemos ensinando o agente de busca a visualizar a página
html_completo_exemplos = """
<main><!--$--><div data-test-target="restaurants-list"><div></div><div class="IDaDx mvTrV cyIij fluiI SMjpI Iwmxp"><div class="f u Q2 qWPrE ncFvv"><a href="/Restaurants-g303404-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ"><svg viewBox="0 0 24 24" width="24px" height="24px" class="d Vb UmNoP" aria-hidden="true"><path d="m16.6 5.65-6.4 6.4 6.4 6.3-1.4 1.4-7.8-7.7 7.8-7.8z"></path></svg><span class="biGQs _P VImYz AWdfh">Restaurantes em Belém</span></a></div></div><div class="IDaDx mvTrV cyIij fluiI SMjpI Iwmxp"><div class="u f k xUqsL mowmC"><div data-test-target="main_h1"><h2 class="biGQs _P CIuBz">Restaurantes internacionais: Belém</h2></div></div></div><div class="IDaDx fluiI SMjpI Iwmxp"><div class="IDaDx mvTrV cyIij fluiI SMjpI Iwmxp"><div class="u f K Q2 ncFvv"><button class="Datwj z W _S Gn B1 Z BF Rd Cj _M j u Nfawr wSSLS XuzGl" type="button" aria-label="Filtros • 1"><div class="SXRwk j u"><span class="Ni"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb UmNoP" aria-hidden="true"><path d="M8 7a1 1 0 1 1-2 0 1 1 0 0 1 2 0m2 10a1 1 0 1 1-2 0 1 1 0 0 1 2 0m8-5a1 1 0 1 1-2 0 1 1 0 0 1 2 0"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M5.418 6.25a1.75 1.75 0 0 1 3.164 0H21v1.5H8.582a1.75 1.75 0 0 1-3.164 0H3v-1.5zm10 5a1.75 1.75 0 0 1 3.164 0H21v1.5h-2.418a1.75 1.75 0 0 1-3.164 0H3v-1.5zm-8 5a1.75 1.75 0 0 1 3.164 0H21v1.5H10.582a1.75 1.75 0 0 1-3.164 0H3v-1.5z"></path></svg></span><span class="biGQs _P FvqAl FGgoA AWdfh">Filtros • 1</span></div></button><div class="H"></div><span data-automation="mapButton"><button class="EAeYu z Xa q W _S Gy Rd Cr _M pGzPy wSSLS" type="button" aria-label="Ver mapa"><span class="sZLBj u k Q1"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M4.25 9.799c0-4.247 3.488-7.707 7.75-7.707s7.75 3.46 7.75 7.707c0 2.28-1.138 4.477-2.471 6.323-1.31 1.813-2.883 3.388-3.977 4.483l-.083.083-.002.002-1.225 1.218-1.213-1.243-.03-.03-.012-.013c-1.1-1.092-2.705-2.687-4.035-4.53-1.324-1.838-2.452-4.024-2.452-6.293M12 3.592c-3.442 0-6.25 2.797-6.25 6.207 0 1.796.907 3.665 2.17 5.415 1.252 1.736 2.778 3.256 3.886 4.357l.043.042.16.164.148-.149.002-.002.061-.06c1.103-1.105 2.605-2.608 3.843-4.322 1.271-1.76 2.187-3.64 2.187-5.445 0-3.41-2.808-6.207-6.25-6.207m1.699 5.013a1.838 1.838 0 1 0-3.397 1.407A1.838 1.838 0 0 0 13.7 8.605m-2.976-2.38a3.338 3.338 0 1 1 2.555 6.168 3.338 3.338 0 0 1-2.555-6.169"></path></svg><span class="biGQs _P AWdfh">Mapa</span></span></button></span></div><div class="u f Q2 mowmC"><div class="biGQs _P VImYz ZNjnF" data-automation="resultsTotal">14 resultados</div><div class="NB"><span><div class="f u"><div class="biGQs _P VImYz egaXP kSNRl ZNjnF">Ordenar por: </div><div class="lOvTR"><button class="UikNM _G B- _S _W u _T c G_ wSSLS wnNQG utTrc" type="button"><span class="biGQs _P ezezH">Relevância</span><div class="CECjK _W PH"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb UmNoP" aria-hidden="true"><path d="M18.4 7.4 12 13.7 5.6 7.4 4.2 8.8l7.8 7.8 7.8-7.8z"></path></svg></div></button></div><div class="GRtXf"><button type="button" class="YEivB _S wSSLS" aria-labelledby="_lithium-r_5_"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb rihVH UmNoP" aria-hidden="true"><path d="M11 10v7h2v-7zm-.034-2.952a1.25 1.25 0 0 0-.216.692A1.24 1.24 0 0 0 12 9a1.25 1.25 0 1 0-1.034-1.952"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M12 2C6.477 2 2 6.477 2 12s4.477 10 10 10 10-4.477 10-10S17.523 2 12 2M4 12a8 8 0 1 1 16 0 8 8 0 0 1-16 0"></path></svg></button><span id="_lithium-r_5_" class="ZVqQo o">Restaurantes classificados de acordo com a correspondência com as suas seleções e avaliações de viajantes.</span></div></div></span></div></div></div><div class="IVEkh Gi"><div class="rfqBq f e"><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d24040497-Reviews-Camarada_Camarao_Belem_Shopping_Boulevard_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4e/d2/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4e/d2/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4e/d2/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4e/d2/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4e/d2/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/92/a0/burger-terra-e-mar.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/92/a0/burger-terra-e-mar.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Burger Terra e Mar" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/92/a0/burger-terra-e-mar.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/92/a0/burger-terra-e-mar.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/92/a0/burger-terra-e-mar.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4f/07/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4f/07/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4f/07/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4f/07/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/2b/4f/07/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/29/23/8f/bd/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/29/23/8f/bd/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/29/23/8f/bd/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/29/23/8f/bd/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/29/23/8f/bd/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/c7/d6/camarada-camarao-boulevard.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/c7/d6/camarada-camarao-boulevard.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Camarada Camarão Boulevard Belém" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/c7/d6/camarada-camarao-boulevard.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/c7/d6/camarada-camarao-boulevard.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/8e/c7/d6/camarada-camarao-boulevard.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/27/14/64/c9/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/27/14/64/c9/camarada-camarao-belem.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/27/14/64/c9/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/27/14/64/c9/camarada-camarao-belem.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/27/14/64/c9/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/22/28/67/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/22/28/67/camarada-camarao-belem.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/22/28/67/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/22/28/67/camarada-camarao-belem.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/25/22/28/67/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/26/2d/66/72/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/26/2d/66/72/camarada-camarao-belem.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/26/2d/66/72/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/26/2d/66/72/camarada-camarao-belem.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/26/2d/66/72/camarada-camarao-belem.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/ed/sobremesa-brownie-com.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/ed/sobremesa-brownie-com.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Sobremesa: Brownie com doce de leite" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/ed/sobremesa-brownie-com.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/ed/sobremesa-brownie-com.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/ed/sobremesa-brownie-com.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/73/refeicao-grelhados-do.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/73/refeicao-grelhados-do.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="Refeição: Grelhados do Chef" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/73/refeicao-grelhados-do.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/73/refeicao-grelhados-do.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/23/74/b0/73/refeicao-grelhados-do.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d24040497-Reviews-Camarada_Camarao_Belem_Shopping_Boulevard_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">1. Camarada Camarão Belém - Shopping Boulevard Belém</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,9</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_6_" data-automation="bubbleRatingImage"><title id="_lithium-r_6_">4,9 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d24040497-Reviews-Camarada_Camarao_Belem_Shopping_Boulevard_Belem-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(1.700 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-24040497"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Brasileira, Frutos do mar</span></span><span class="biGQs _P VImYz ZNjnF">$$ - $$$</span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f lFNDg"><span class="biGQs _P ZNjnF">Aberto agora</span></div></span><span class="f"><a target="_blank" href="https://livemenu.app/menu/627bcbac218bd50011827877" class="BMQDV _F Gv wSSLS SwZTJ"><span class="SZrBo"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M16.719 8.193V19.97h-1.5v-4.124h-1.987V10.94c0-1.03.453-1.748 1.055-2.184a3 3 0 0 1 1.67-.552zm-1.5 6.153v-4.41l-.052.036c-.226.164-.435.435-.435.97v3.404z"></path><path d="M9.086 19.97h1.5v-7.72q.368-.094.672-.295c.382-.254.633-.6.796-.957.31-.682.337-1.488.337-2.043v-.75h-1.5v.75c0 .556-.04 1.065-.202 1.42a1 1 0 0 1-.103.178V8.205h-1.5v2.28a1 1 0 0 1-.068-.118c-.176-.344-.237-.85-.237-1.412v-.75h-1.5v.75c0 .604.054 1.414.4 2.093.18.353.446.686.833.927q.264.163.572.253z"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M18.817.317 4.136 4.492H4.02v19.191h15.96V4.492h-1.162zm-1.5 4.175v-2.19L9.62 4.493zm1.162 1.5H5.52v16.191h12.96z"></path></svg><span class="biGQs _P VImYz ZNjnF">Cardápio</span></span></a></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d24040497-Reviews-Camarada_Camarao_Belem_Shopping_Boulevard_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Maravilhosa</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d24040497-Reviews-Camarada_Camarao_Belem_Shopping_Boulevard_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Maravilhosa</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d14107327-Reviews-Restaurante_JP_Grill-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/ff/59/63/photo0jpg.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/ff/59/63/photo0jpg.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/ff/59/63/photo0jpg.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/ff/59/63/photo0jpg.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/ff/59/63/photo0jpg.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a3/photo1jpg.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a3/photo1jpg.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a3/photo1jpg.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a3/photo1jpg.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a3/photo1jpg.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/9f/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/9f/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/9f/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/9f/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/9f/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/a0/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/a0/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/a0/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/a0/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/31/e4/d9/a0/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a1/photo0jpg.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a1/photo0jpg.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a1/photo0jpg.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a1/photo0jpg.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/15/05/30/a1/photo0jpg.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d14107327-Reviews-Restaurante_JP_Grill-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">2. Restaurante JP Grill</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,8</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_f_" data-automation="bubbleRatingImage"><title id="_lithium-r_f_">4,8 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d14107327-Reviews-Restaurante_JP_Grill-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(12 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-14107327"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Brasileira, Internacional</span></span><span class="biGQs _P VImYz ZNjnF">$$ - $$$</span></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d14107327-Reviews-Restaurante_JP_Grill-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Vale o que custo</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d14107327-Reviews-Restaurante_JP_Grill-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Restaurante JP</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d21311161-Reviews-Mirakuru_Street_Food-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/1f/3f/28/e-bem-desse-jeito-mesmo.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/1f/3f/28/e-bem-desse-jeito-mesmo.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="É bem desse jeito mesmo: calçada esburacada e parede faltando reboco, mas garantimos que a comida é de qualidade." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/1f/3f/28/e-bem-desse-jeito-mesmo.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/1f/3f/28/e-bem-desse-jeito-mesmo.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/1f/3f/28/e-bem-desse-jeito-mesmo.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/3a/tantanmen-macarrao-caseiro.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/3a/tantanmen-macarrao-caseiro.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="Tantanmen - Macarrão caseiro, caldo de porco, ovo curado no shoyu, carne de pouco moída, cebolinha e bok choi" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/3a/tantanmen-macarrao-caseiro.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/3a/tantanmen-macarrao-caseiro.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/3a/tantanmen-macarrao-caseiro.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4b/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4b/chashu-shio-ramen-macarrao.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="Chashu Shio Ramen - Macarrão caseiro, caldo de frango com porco, porco assado, milho queimado, cebolinha, ovo curado no shoyu" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4b/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4b/chashu-shio-ramen-macarrao.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4b/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/64/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/64/chashu-shio-ramen-macarrao.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Chashu Shio Ramen - Macarrão caseiro, caldo de frango com porco, porco assado, milho queimado, cebolinha, ovo curado no shoyu" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/64/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/64/chashu-shio-ramen-macarrao.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/64/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/79/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/79/chashu-shio-ramen-macarrao.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Chashu Shio Ramen - Macarrão caseiro, caldo de frango com porco, porco assado, milho queimado, cebolinha, ovo curado no shoyu" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/79/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/79/chashu-shio-ramen-macarrao.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/79/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/75/tonkatsu-yakisoba-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/75/tonkatsu-yakisoba-macarrao.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Tonkatsu Yakisoba - Macarrão caseiro, mix de legumes, porco frito, maionese kewpie, molho tonkatsu, molho de yakisoba caseiro, gengibre em conserva" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/75/tonkatsu-yakisoba-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/75/tonkatsu-yakisoba-macarrao.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/75/tonkatsu-yakisoba-macarrao.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/36/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/36/chashu-shio-ramen-macarrao.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="Chashu Shio Ramen - Macarrão caseiro, caldo de frango com porco, porco assado, pickles de nabo, cebolinha, ovo curado no shoyu" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/36/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/36/chashu-shio-ramen-macarrao.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/36/chashu-shio-ramen-macarrao.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/38/tonkatsu-shoyu-ramen.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/38/tonkatsu-shoyu-ramen.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Tonkatsu Shoyu Ramen - Macarrão caseiro, caldo de frango com porco e shoyu, porco empanado, pickles de nabo, cebolinha e ovo curado no shoyu" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/38/tonkatsu-shoyu-ramen.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/38/tonkatsu-shoyu-ramen.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/38/tonkatsu-shoyu-ramen.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/60/karaage-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/60/karaage-shio-ramen-macarrao.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Karaage Shio Ramen - Macarrão caseiro, caldo de frango com porco, frango frito, repolho assado, cebolinha, ovo curado no shoyu" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/60/karaage-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/60/karaage-shio-ramen-macarrao.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/60/karaage-shio-ramen-macarrao.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4f/pho-bo-caldo-de-carne.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4f/pho-bo-caldo-de-carne.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="Pho Bo - Caldo de carne com especiarias chinesas, tutano assado, flat iron, cebolinha, coentro e broto de feijão" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4f/pho-bo-caldo-de-carne.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4f/pho-bo-caldo-de-carne.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1c/24/c3/4f/pho-bo-caldo-de-carne.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d21311161-Reviews-Mirakuru_Street_Food-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">3. Mirakuru Street Food</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>5,0</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_o_" data-automation="bubbleRatingImage"><title id="_lithium-r_o_">5 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d21311161-Reviews-Mirakuru_Street_Food-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(4 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-21311161"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Internacional, Asiática</span></span><span class="biGQs _P VImYz ZNjnF">$</span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f RAyMq"><span class="biGQs _P ZNjnF">Fechado agora</span></div></span><span class="f"><a target="_blank" href="https://mirakurubelem.wixsite.com/website" class="BMQDV _F Gv wSSLS SwZTJ"><span class="SZrBo"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M16.719 8.193V19.97h-1.5v-4.124h-1.987V10.94c0-1.03.453-1.748 1.055-2.184a3 3 0 0 1 1.67-.552zm-1.5 6.153v-4.41l-.052.036c-.226.164-.435.435-.435.97v3.404z"></path><path d="M9.086 19.97h1.5v-7.72q.368-.094.672-.295c.382-.254.633-.6.796-.957.31-.682.337-1.488.337-2.043v-.75h-1.5v.75c0 .556-.04 1.065-.202 1.42a1 1 0 0 1-.103.178V8.205h-1.5v2.28a1 1 0 0 1-.068-.118c-.176-.344-.237-.85-.237-1.412v-.75h-1.5v.75c0 .604.054 1.414.4 2.093.18.353.446.686.833.927q.264.163.572.253z"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M18.817.317 4.136 4.492H4.02v19.191h15.96V4.492h-1.162zm-1.5 4.175v-2.19L9.62 4.493zm1.162 1.5H5.52v16.191h12.96z"></path></svg><span class="biGQs _P VImYz ZNjnF">Cardápio</span></span></a></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d21311161-Reviews-Mirakuru_Street_Food-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Comida excelente</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d21311161-Reviews-Mirakuru_Street_Food-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Experiência que vale à pena.</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div class=""><div id="slot:300x250-8x8:inline1" class="Eufjb j u ChFsW Xd o S nFcpI"><div id="google_ads_iframe_/5349/ta.ta.br.s/sa.brazil.state_of_para_6__container__" style="border: 0pt none; display: inline-block;"></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d26491778-Reviews-Old_House_Blues_e_Jazz-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6f/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6f/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6f/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6f/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6f/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/75/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/75/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/75/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/75/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/75/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/76/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/76/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/76/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/76/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/76/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2b/6b/f2/10/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2b/6b/f2/10/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2b/6b/f2/10/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2b/6b/f2/10/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2b/6b/f2/10/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/43/c7/adega-casa-blanca.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/43/c7/adega-casa-blanca.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Adega Casa Blanca" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/43/c7/adega-casa-blanca.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/43/c7/adega-casa-blanca.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/43/c7/adega-casa-blanca.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/4c/fa/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/4c/fa/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/4c/fa/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/4c/fa/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/8b/4c/fa/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6a/gastronomia.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6a/gastronomia.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="GASTRONOMIA " fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6a/gastronomia.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6a/gastronomia.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/6a/gastronomia.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/72/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/72/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/72/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/72/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/72/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/70/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/70/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/70/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/70/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/70/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/71/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/71/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/71/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/71/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/71/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/77/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/77/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/77/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/77/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2d/f0/01/77/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d26491778-Reviews-Old_House_Blues_e_Jazz-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">4. Old House - Blues e Jazz</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,2</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_11_" data-automation="bubbleRatingImage"><title id="_lithium-r_11_">4,2 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d26491778-Reviews-Old_House_Blues_e_Jazz-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(11 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-26491778"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Italiana, Bar</span></span><span class="biGQs _P VImYz ZNjnF">$$$$</span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f RAyMq"><span class="biGQs _P ZNjnF">Fechado agora</span></div></span><span class="f"><button type="button" class="LuHAv Cj G_ B-"><span class="SZrBo"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M16.719 8.193V19.97h-1.5v-4.124h-1.987V10.94c0-1.03.453-1.748 1.055-2.184a3 3 0 0 1 1.67-.552zm-1.5 6.153v-4.41l-.052.036c-.226.164-.435.435-.435.97v3.404z"></path><path d="M9.086 19.97h1.5v-7.72q.368-.094.672-.295c.382-.254.633-.6.796-.957.31-.682.337-1.488.337-2.043v-.75h-1.5v.75c0 .556-.04 1.065-.202 1.42a1 1 0 0 1-.103.178V8.205h-1.5v2.28a1 1 0 0 1-.068-.118c-.176-.344-.237-.85-.237-1.412v-.75h-1.5v.75c0 .604.054 1.414.4 2.093.18.353.446.686.833.927q.264.163.572.253z"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M18.817.317 4.136 4.492H4.02v19.191h15.96V4.492h-1.162zm-1.5 4.175v-2.19L9.62 4.493zm1.162 1.5H5.52v16.191h12.96z"></path></svg><span class="biGQs _P VImYz ZNjnF">Cardápio</span></span></button></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d26491778-Reviews-Old_House_Blues_e_Jazz-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Muito aconchegante</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d26491778-Reviews-Old_House_Blues_e_Jazz-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Experiência única em Belém</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d7252312-Reviews-Radisson_Hotel_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/5f/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/5f/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Quarto imenso" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/5f/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/5f/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/5f/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/50/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/50/caption.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Quarto espaçoso." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/50/caption.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/50/caption.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/2a/26/5c/50/caption.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d7252312-Reviews-Radisson_Hotel_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">5. Radisson Hotel Belem</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>3,9</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_1a_" data-automation="bubbleRatingImage"><title id="_lithium-r_1a_">3,9 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d7252312-Reviews-Radisson_Hotel_Belem-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(9 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-7252312"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Internacional, Europeia</span></span><span class="biGQs _P VImYz ZNjnF"></span></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d7252312-Reviews-Radisson_Hotel_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Maravilhoso!</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d7252312-Reviews-Radisson_Hotel_Belem-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Magnífico </span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d2373296-Reviews-Na_Telha-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/b0/e1/39/20180104-211513-largejpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/b0/e1/39/20180104-211513-largejpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/b0/e1/39/20180104-211513-largejpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/b0/e1/39/20180104-211513-largejpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/b0/e1/39/20180104-211513-largejpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0f/ec/6f/4d/photo0jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0f/ec/6f/4d/photo0jpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0f/ec/6f/4d/photo0jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0f/ec/6f/4d/photo0jpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0f/ec/6f/4d/photo0jpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/e1/f1/c8/20151121-140123-largejpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/e1/f1/c8/20151121-140123-largejpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/e1/f1/c8/20151121-140123-largejpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/e1/f1/c8/20151121-140123-largejpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/e1/f1/c8/20151121-140123-largejpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/61/photo1jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/61/photo1jpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/61/photo1jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/61/photo1jpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/61/photo1jpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/d8/8b/72/na-telha-de-peixe-e-camarao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/d8/8b/72/na-telha-de-peixe-e-camarao.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Na telha de peixe e camarão.  Caldeirada." fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/d8/8b/72/na-telha-de-peixe-e-camarao.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/d8/8b/72/na-telha-de-peixe-e-camarao.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/d8/8b/72/na-telha-de-peixe-e-camarao.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/60/photo0jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/60/photo0jpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/60/photo0jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/60/photo0jpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/10/5c/75/60/photo0jpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/88/photo1jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/88/photo1jpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/88/photo1jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/88/photo1jpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/88/photo1jpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/87/photo0jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/87/photo0jpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/87/photo0jpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/87/photo0jpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/16/71/c0/87/photo0jpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/bc/69/b6/caldeirada-muito-saborosa.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/bc/69/b6/caldeirada-muito-saborosa.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Caldeirada muito saborosa!" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/bc/69/b6/caldeirada-muito-saborosa.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/bc/69/b6/caldeirada-muito-saborosa.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/bc/69/b6/caldeirada-muito-saborosa.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/94/1a/fb/peixe-na-telha.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/94/1a/fb/peixe-na-telha.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Peixe na telha!" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/94/1a/fb/peixe-na-telha.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/94/1a/fb/peixe-na-telha.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/08/94/1a/fb/peixe-na-telha.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0c/fd/a3/9a/na-telha.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0c/fd/a3/9a/na-telha.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="Na Telha" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0c/fd/a3/9a/na-telha.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0c/fd/a3/9a/na-telha.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0c/fd/a3/9a/na-telha.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d2373296-Reviews-Na_Telha-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">6. Na Telha</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,2</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_1j_" data-automation="bubbleRatingImage"><title id="_lithium-r_1j_">4,2 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d2373296-Reviews-Na_Telha-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(142 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-2373296"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Brasileira, Frutos do mar</span></span><span class="biGQs _P VImYz ZNjnF">$$ - $$$</span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f lFNDg"><span class="biGQs _P ZNjnF">Aberto agora</span></div></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d2373296-Reviews-Na_Telha-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Almoço </span></div></a><a target="_self" href="/Restaurant_Review-g303404-d2373296-Reviews-Na_Telha-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Vale a experiência! </span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d9558565-Reviews-Restaurante_Casa_Mia-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/22/4f/a9/df/fachada-da-nossa-casa.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/22/4f/a9/df/fachada-da-nossa-casa.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/22/4f/a9/df/fachada-da-nossa-casa.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/22/4f/a9/df/fachada-da-nossa-casa.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/22/4f/a9/df/fachada-da-nossa-casa.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d9558565-Reviews-Restaurante_Casa_Mia-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">7. Restaurante Casa Mia</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,0</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_1s_" data-automation="bubbleRatingImage"><title id="_lithium-r_1s_">4 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d9558565-Reviews-Restaurante_Casa_Mia-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(167 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-9558565"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Brasileira, Café</span></span><span class="biGQs _P VImYz ZNjnF">$$ - $$$</span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f lFNDg"><span class="biGQs _P ZNjnF">Aberto agora</span></div></span><span class="f"><a target="_blank" href="https://cdn.flipsnack.com/widget/v2/widget.html?hash=t30v5s5hp8" class="BMQDV _F Gv wSSLS SwZTJ"><span class="SZrBo"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M16.719 8.193V19.97h-1.5v-4.124h-1.987V10.94c0-1.03.453-1.748 1.055-2.184a3 3 0 0 1 1.67-.552zm-1.5 6.153v-4.41l-.052.036c-.226.164-.435.435-.435.97v3.404z"></path><path d="M9.086 19.97h1.5v-7.72q.368-.094.672-.295c.382-.254.633-.6.796-.957.31-.682.337-1.488.337-2.043v-.75h-1.5v.75c0 .556-.04 1.065-.202 1.42a1 1 0 0 1-.103.178V8.205h-1.5v2.28a1 1 0 0 1-.068-.118c-.176-.344-.237-.85-.237-1.412v-.75h-1.5v.75c0 .604.054 1.414.4 2.093.18.353.446.686.833.927q.264.163.572.253z"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M18.817.317 4.136 4.492H4.02v19.191h15.96V4.492h-1.162zm-1.5 4.175v-2.19L9.62 4.493zm1.162 1.5H5.52v16.191h12.96z"></path></svg><span class="biGQs _P VImYz ZNjnF">Cardápio</span></span></a></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d9558565-Reviews-Restaurante_Casa_Mia-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">comida boa, atendimento demorado</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d9558565-Reviews-Restaurante_Casa_Mia-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Bom restaurante com área kids</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div class=""><div id="slot:300x250-8x8:inline2" class="Eufjb j u ChFsW Xd o S nFcpI"><div id="google_ads_iframe_/5349/ta.ta.br.s/sa.brazil.state_of_para_7__container__" style="border: 0pt none; display: inline-block;"></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d2373295-Reviews-Estacao_Gourmet-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/19/89/34/estacao-gourmet.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/19/89/34/estacao-gourmet.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/19/89/34/estacao-gourmet.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/19/89/34/estacao-gourmet.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/06/19/89/34/estacao-gourmet.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d2373295-Reviews-Estacao_Gourmet-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">8. Estação Gourmet</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,5</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_25_" data-automation="bubbleRatingImage"><title id="_lithium-r_25_">4,5 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.389 0 0 5.389 0 12c0 6.62 5.389 12 12 12 6.62 0 12-5.379 12-12S18.621 0 12 0zm0 2a9.984 9.984 0 0110 10 9.976 9.976 0 01-10 10z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d2373295-Reviews-Estacao_Gourmet-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(83 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-2373295"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Brasileira, Internacional</span></span><span class="biGQs _P VImYz ZNjnF">$$ - $$$</span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f lFNDg"><span class="biGQs _P ZNjnF">Aberto agora</span></div></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d2373295-Reviews-Estacao_Gourmet-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Supimpa </span></div></a><a target="_self" href="/Restaurant_Review-g303404-d2373295-Reviews-Estacao_Gourmet-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Excelente opção de comida a quilo em Belém</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d10480155-Reviews-Vegas_Poker_Club-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/a2/b1/f9/20171229-183247-largejpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/a2/b1/f9/20171229-183247-largejpg.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/a2/b1/f9/20171229-183247-largejpg.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/a2/b1/f9/20171229-183247-largejpg.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/11/a2/b1/f9/20171229-183247-largejpg.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d10480155-Reviews-Vegas_Poker_Club-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">9. Vegas Poker Club</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>4,1</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_2e_" data-automation="bubbleRatingImage"><title id="_lithium-r_2e_">4,1 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d10480155-Reviews-Vegas_Poker_Club-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(38 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-10480155"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Brasileira, Bar</span></span><span class="biGQs _P VImYz ZNjnF">$$ - $$$</span></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d10480155-Reviews-Vegas_Poker_Club-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Escutar uma boa música tomando um chopp!</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d10480155-Reviews-Vegas_Poker_Club-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Legal para tomar um chopp</span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d12316068-Reviews-Le_Bistrot-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/13/02/d5/e9/nossa-logomarca-desde.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/13/02/d5/e9/nossa-logomarca-desde.jpg?w=300&amp;h=-1&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/13/02/d5/e9/nossa-logomarca-desde.jpg?w=200&amp;h=-1&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/13/02/d5/e9/nossa-logomarca-desde.jpg?w=400&amp;h=-1&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/13/02/d5/e9/nossa-logomarca-desde.jpg?w=200&amp;h=-1&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d12316068-Reviews-Le_Bistrot-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">10. Le Bistrot</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><span aria-hidden="true"><div class="biGQs _P VImYz ZNjnF" data-automation="bubbleRatingValue"><span>3,7</span></div></span><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_2n_" data-automation="bubbleRatingImage"><title id="_lithium-r_2n_">3,7 de 5 círculos</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12z" transform="translate(52 0)"></path><path d="M 12 0C5.389 0 0 5.389 0 12c0 6.62 5.389 12 12 12 6.62 0 12-5.379 12-12S18.621 0 12 0zm0 2a9.984 9.984 0 0110 10 9.976 9.976 0 01-10 10z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d12316068-Reviews-Le_Bistrot-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(26 avaliações) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-12316068"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Francesa, Internacional</span></span><span class="biGQs _P VImYz ZNjnF"></span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f RAyMq"><span class="biGQs _P ZNjnF">Fechado agora</span></div></span></div></div></div></div></div></div></div><div class="syfsD o"><div class="EURYK" tabindex="0" role="button"><div class="MUVLp f e"><a target="_self" href="/Restaurant_Review-g303404-d12316068-Reviews-Le_Bistrot-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Um lugar acolhedor</span></div></a><a target="_self" href="/Restaurant_Review-g303404-d12316068-Reviews-Le_Bistrot-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P VImYz alXOW EEXWj GzNcM BYtua UTQMg alvrA ZNjnF"><span class="oTOkl">Boa opção em Belém </span></div></a></div></div></div><div class="NJIJw"></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d13435710-Reviews-Bar_e_Restaurante_La_Marine-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="yMcGX"><picture class="NhWcC _R mdkdE afQPz eXZKw"><source media="(max-width: 767px)" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/14/b4/56/dd/voce-pode-almocar-em.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/14/b4/56/dd/voce-pode-almocar-em.jpg?w=300&amp;h=300&amp;s=1 2x"><img width="200" height="200" alt="" fetchpriority="high" srcset="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/14/b4/56/dd/voce-pode-almocar-em.jpg?w=200&amp;h=200&amp;s=1 1x,https://dynamic-media-cdn.tripadvisor.com/media/photo-o/14/b4/56/dd/voce-pode-almocar-em.jpg?w=400&amp;h=400&amp;s=1 2x" src="https://dynamic-media-cdn.tripadvisor.com/media/photo-o/14/b4/56/dd/voce-pode-almocar-em.jpg?w=200&amp;h=200&amp;s=1"></picture></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d13435710-Reviews-Bar_e_Restaurante_La_Marine-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">11. Bar e Restaurante La Marine</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_30_" data-automation="bubbleRatingImage"><title id="_lithium-r_30_">Ainda não há avaliações</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d13435710-Reviews-Bar_e_Restaurante_La_Marine-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(0 avaliação) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-13435710"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Francesa, Brasileira</span></span><span class="biGQs _P VImYz ZNjnF">$</span></div></div></div></div></div></div><div class="NJIJw"><div class="bbfVZ" tabindex="0" role="button"><a class="rmyCe _G B- z _S c Wc wSSLS jWkoZ QHaGY" href="/UserReviewEdit-g303404-d13435710-Reviews-Bar_e_Restaurante_La_Marine-Belem_State_of_Para.html"><span class="biGQs _P ezezH"><div class="tWvGj"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M13.788 7.293 6.19 14.868l-.816 3.746 3.724-.839 7.588-7.583zm3.96 1.84-2.898-2.9.556-.554A2.32 2.32 0 0 1 17.02 5c.378 0 .73.104 1.031.315l.01.007.012.008c1.12.757 1.221 2.26.326 3.151zm-7.896 10.01-5.99 1.35q-.032.009-.064.007a.297.297 0 0 1-.29-.36l1.31-6.023 9.529-9.5A3.82 3.82 0 0 1 17.02 3.5c.66 0 1.318.184 1.893.587a3.536 3.536 0 0 1 .546 5.457z"></path></svg>Faça uma avaliação </div></span></a></div></div></div></div></div><div class=""><div id="slot:300x250-8x8:inline3" class="Eufjb j u ChFsW Xd o S nFcpI"><div id="google_ads_iframe_/5349/ta.ta.br.s/sa.brazil.state_of_para_8__container__" style="border: 0pt none; display: inline-block;"></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d27167379-Reviews-Dalie_Mediterrane-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><svg xmlns="http://www.w3.org/2000/svg" class="QgHgJ"><svg x="50%" y="50%" width="50" height="50" viewBox="0 0 40 40" xmlns="http://www.w3.org/2000/svg" overflow="visible"><g transform="translate(-20, -20)"><path d="M28.8 4.53333C32.4819 4.53333 35.4667 7.5181 35.4667 11.2C35.4667 14.8819 32.4819 17.8667 28.8 17.8667C25.1181 17.8667 22.1333 14.8819 22.1333 11.2C22.1333 7.5181 25.1181 4.53333 28.8 4.53333Z" fill="var(--scheme-inverse-on-surface)"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M37.5717 0.00078125C38.9186 0.034925 40 1.13748 40 2.49258V37.5074L39.9992 37.5717C39.9656 38.8972 38.8972 39.9656 37.5717 39.9992L37.5074 40H2.49258L2.42826 39.9992C1.10276 39.9656 0.0343837 38.8972 0.00078125 37.5717L0 37.5074V2.49258C2.78546e-05 1.13748 1.08138 0.0349255 2.42826 0.00078125L2.49258 0H37.5074L37.5717 0.00078125ZM2.49258 2.4C2.44146 2.40003 2.40003 2.44146 2.4 2.49258V23.4285C4.22877 21.3137 6.98148 19.4667 10.6667 19.4667C17.9151 19.4667 19.4369 29.2834 37.6 30.0815V2.49258C37.6 2.44146 37.5585 2.40003 37.5074 2.4H2.49258Z" fill="var(--scheme-inverse-on-surface)"></path></g></svg></svg></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d27167379-Reviews-Dalie_Mediterrane-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">12. Dalie Mediterrané</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_39_" data-automation="bubbleRatingImage"><title id="_lithium-r_39_">Ainda não há avaliações</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d27167379-Reviews-Dalie_Mediterrane-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(0 avaliação) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-27167379"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Internacional, Mediterrânea</span></span><span class="biGQs _P VImYz ZNjnF"></span></div></div></div></div></div></div><div class="NJIJw"><div class="bbfVZ" tabindex="0" role="button"><a class="rmyCe _G B- z _S c Wc wSSLS jWkoZ QHaGY" href="/UserReviewEdit-g303404-d27167379-Reviews-Dalie_Mediterrane-Belem_State_of_Para.html"><span class="biGQs _P ezezH"><div class="tWvGj"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M13.788 7.293 6.19 14.868l-.816 3.746 3.724-.839 7.588-7.583zm3.96 1.84-2.898-2.9.556-.554A2.32 2.32 0 0 1 17.02 5c.378 0 .73.104 1.031.315l.01.007.012.008c1.12.757 1.221 2.26.326 3.151zm-7.896 10.01-5.99 1.35q-.032.009-.064.007a.297.297 0 0 1-.29-.36l1.31-6.023 9.529-9.5A3.82 3.82 0 0 1 17.02 3.5c.66 0 1.318.184 1.893.587a3.536 3.536 0 0 1 .546 5.457z"></path></svg>Faça uma avaliação </div></span></a></div></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d33354040-Reviews-The_Coffee_Batista_Campos-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><svg xmlns="http://www.w3.org/2000/svg" class="QgHgJ"><svg x="50%" y="50%" width="50" height="50" viewBox="0 0 40 40" xmlns="http://www.w3.org/2000/svg" overflow="visible"><g transform="translate(-20, -20)"><path d="M28.8 4.53333C32.4819 4.53333 35.4667 7.5181 35.4667 11.2C35.4667 14.8819 32.4819 17.8667 28.8 17.8667C25.1181 17.8667 22.1333 14.8819 22.1333 11.2C22.1333 7.5181 25.1181 4.53333 28.8 4.53333Z" fill="var(--scheme-inverse-on-surface)"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M37.5717 0.00078125C38.9186 0.034925 40 1.13748 40 2.49258V37.5074L39.9992 37.5717C39.9656 38.8972 38.8972 39.9656 37.5717 39.9992L37.5074 40H2.49258L2.42826 39.9992C1.10276 39.9656 0.0343837 38.8972 0.00078125 37.5717L0 37.5074V2.49258C2.78546e-05 1.13748 1.08138 0.0349255 2.42826 0.00078125L2.49258 0H37.5074L37.5717 0.00078125ZM2.49258 2.4C2.44146 2.40003 2.40003 2.44146 2.4 2.49258V23.4285C4.22877 21.3137 6.98148 19.4667 10.6667 19.4667C17.9151 19.4667 19.4369 29.2834 37.6 30.0815V2.49258C37.6 2.44146 37.5585 2.40003 37.5074 2.4H2.49258Z" fill="var(--scheme-inverse-on-surface)"></path></g></svg></svg></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d33354040-Reviews-The_Coffee_Batista_Campos-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">13. The Coffee Batista Campos</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_3i_" data-automation="bubbleRatingImage"><title id="_lithium-r_3i_">Ainda não há avaliações</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d33354040-Reviews-The_Coffee_Batista_Campos-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(0 avaliação) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-33354040"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Café e chá, Japonesa</span></span><span class="biGQs _P VImYz ZNjnF"></span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f lFNDg"><span class="biGQs _P ZNjnF">Aberto agora</span></div></span></div></div></div></div></div></div></div><div class="NJIJw"><div class="bbfVZ" tabindex="0" role="button"><a class="rmyCe _G B- z _S c Wc wSSLS jWkoZ QHaGY" href="/UserReviewEdit-g303404-d33354040-Reviews-The_Coffee_Batista_Campos-Belem_State_of_Para.html"><span class="biGQs _P ezezH"><div class="tWvGj"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M13.788 7.293 6.19 14.868l-.816 3.746 3.724-.839 7.588-7.583zm3.96 1.84-2.898-2.9.556-.554A2.32 2.32 0 0 1 17.02 5c.378 0 .73.104 1.031.315l.01.007.012.008c1.12.757 1.221 2.26.326 3.151zm-7.896 10.01-5.99 1.35q-.032.009-.064.007a.297.297 0 0 1-.29-.36l1.31-6.023 9.529-9.5A3.82 3.82 0 0 1 17.02 3.5c.66 0 1.318.184 1.893.587a3.536 3.536 0 0 1 .546 5.457z"></path></svg>Faça uma avaliação </div></span></a></div></div></div></div></div><div data-automation="restaurantCard"><div class="FpUvB Ft MA w Re Gi"><div class="kooPS B1 Z BB Re Gi"><div class="lZkVi w o"><div></div><a href="/Restaurant_Review-g303404-d24941804-Reviews-Delicatesse-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt PaRlG"><div class="WcLPg _Z Mk NL"><div class="_T w _Z" data-clicksource="Photo"><div><div class="qfYLD _T"><div class="kjQGJ"><div class="acVWG _t z"><ul class="qRYpc"><li class="GiSaP Fg I _u"><div class="Re o _Z"><svg xmlns="http://www.w3.org/2000/svg" class="QgHgJ"><svg x="50%" y="50%" width="50" height="50" viewBox="0 0 40 40" xmlns="http://www.w3.org/2000/svg" overflow="visible"><g transform="translate(-20, -20)"><path d="M28.8 4.53333C32.4819 4.53333 35.4667 7.5181 35.4667 11.2C35.4667 14.8819 32.4819 17.8667 28.8 17.8667C25.1181 17.8667 22.1333 14.8819 22.1333 11.2C22.1333 7.5181 25.1181 4.53333 28.8 4.53333Z" fill="var(--scheme-inverse-on-surface)"></path><path fill-rule="evenodd" clip-rule="evenodd" d="M37.5717 0.00078125C38.9186 0.034925 40 1.13748 40 2.49258V37.5074L39.9992 37.5717C39.9656 38.8972 38.8972 39.9656 37.5717 39.9992L37.5074 40H2.49258L2.42826 39.9992C1.10276 39.9656 0.0343837 38.8972 0.00078125 37.5717L0 37.5074V2.49258C2.78546e-05 1.13748 1.08138 0.0349255 2.42826 0.00078125L2.49258 0H37.5074L37.5717 0.00078125ZM2.49258 2.4C2.44146 2.40003 2.40003 2.44146 2.4 2.49258V23.4285C4.22877 21.3137 6.98148 19.4667 10.6667 19.4667C17.9151 19.4667 19.4369 29.2834 37.6 30.0815V2.49258C37.6 2.44146 37.5585 2.40003 37.5074 2.4H2.49258Z" fill="var(--scheme-inverse-on-surface)"></path></g></svg></svg></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li><li class="GiSaP Fg I _u"><div class="Re o _Z"><div class="tMjsv w _Z Gz" data-skeleton="@ta/design-system.skeleton" style="display: block;"></div></div><div data-carousel-waypoint="true"></div></li></ul></div></div></div></div></div></div></a></div><div class="YEIJb"><div tabindex="0" role="button"><div class="mfKvs f e"><div class="UIwAG f k"><div class="KBZbF f e"><div class="ZvrsW N G"><a href="/Restaurant_Review-g303404-d24941804-Reviews-Delicatesse-Belem_State_of_Para.html" class="BMQDV _F Gv wSSLS SwZTJ FGwzt ukgoS"><div class="biGQs _P SewaP alXOW oCpZu GzNcM nvOhm UTQMg ZTpaU mtnKn OgHoE">14. Délicatesse</div></a><div class="kzrsh"><div class="VVbkp"><div class="MyMKp u Q1"><div class="nKWJn u Q1 qMyjI"><svg class="evwcZ" viewBox="0 0 128 24" width="68" height="12" aria-labelledby="_lithium-r_3r_" data-automation="bubbleRatingImage"><title id="_lithium-r_3r_">Ainda não há avaliações</title><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform=""></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(26 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(52 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(78 0)"></path><path d="M 12 0C5.388 0 0 5.388 0 12s5.388 12 12 12 12-5.38 12-12c0-6.612-5.38-12-12-12zm0 2a9.983 9.983 0 019.995 10 10 10 0 01-10 10A10 10 0 012 12 10 10 0 0112 2z" transform="translate(104 0)"></path></svg><div class="biGQs _P VImYz ZNjnF"><a target="_self" href="/Restaurant_Review-g303404-d24941804-Reviews-Delicatesse-Belem_State_of_Para.html#REVIEWS" class="BMQDV _F Gv wSSLS SwZTJ FGwzt suezE"><div class="biGQs _P ZNjnF" data-automation="bubbleReviewCount"><span>(0 avaliação) </span></div></a></div></div></div></div></div></div></div><div class="Wrtgu"><button type="button" aria-label="Salvar em uma viagem" class="vKDXe u j _T z _S _F xwbQG cunPs jODdB" data-automation="trips-save-button-item-24941804"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb icjEL" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M3.798 5.166A5.77 5.77 0 0 1 7.72 3.63c1.455 0 2.857.548 3.922 1.536l.005.005.341.322.332-.317a5.77 5.77 0 0 1 3.928-1.54c1.458 0 2.862.55 3.928 1.54l.004.004c1.093 1.032 1.598 2.324 1.569 3.662-.03 1.323-.579 2.643-1.5 3.785-.884 1.096-2.85 2.943-4.547 4.478a185 185 0 0 1-3.153 2.785l-.069.059-.489-.569.489.569-.485.416-.488-.412a102 102 0 0 1-7.75-7.288l-.021-.021-.02-.023c-1.725-2.115-2.203-5.32.08-7.453zm8.19 13.226.472-.412a184 184 0 0 0 2.236-1.988c1.72-1.556 3.59-3.32 4.385-4.306.757-.939 1.147-1.948 1.168-2.877.02-.912-.313-1.795-1.097-2.536a4.27 4.27 0 0 0-2.904-1.138c-1.08 0-2.117.407-2.903 1.136l-1.35 1.292-1.375-1.3a4.27 4.27 0 0 0-2.9-1.133 4.27 4.27 0 0 0-2.901 1.135c-1.507 1.408-1.353 3.659.042 5.385a101 101 0 0 0 7.127 6.742"></path></svg></button></div></div><div class="Mi"><div class="ZvrsW N G"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M14.051 6.549v.003l1.134 1.14 3.241-3.25.003-.002 1.134 1.136-3.243 3.252 1.134 1.14a1 1 0 0 0 .09-.008c.293-.05.573-.324.72-.474l.005-.006 2.596-2.603L22 8.016l-2.597 2.604a3.73 3.73 0 0 1-1.982 1.015 4.3 4.3 0 0 1-3.162-.657l-.023-.016-.026-.018-1.366 1.407 8.509 8.512L20.219 22l-.002-.002-6.654-6.663-2.597 2.76-7.3-7.315C1.967 8.948 1.531 6.274 2.524 4.198c.241-.504.566-.973.978-1.386l8.154 8.416 1.418-1.423-.039-.045c-.858-1.002-1.048-2.368-.62-3.595a4.15 4.15 0 0 1 .983-1.561L16 2l1.135 1.138-2.598 2.602-.047.045c-.16.151-.394.374-.433.678zM3.809 5.523c-.362 1.319-.037 2.905 1.06 4.103L10.93 15.7l1.408-1.496zM2.205 20.697 3.34 21.84l4.543-4.552-1.135-1.143z"></path></svg><span class="biGQs _P VImYz ZNjnF">Steakhouse, Brasileira</span></span><span class="biGQs _P VImYz ZNjnF"></span></div><div class="Mi cJyep f k K"><div class="ZvrsW N G biqBm"><span class="f"><svg viewBox="0 0 24 24" width="16px" height="16px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M12 3.75a8.25 8.25 0 1 0 0 16.5 8.25 8.25 0 0 0 0-16.5M2.25 12c0-5.385 4.365-9.75 9.75-9.75s9.75 4.365 9.75 9.75-4.365 9.75-9.75 9.75S2.25 17.385 2.25 12m9-5.5h1.5v5.19l3.28 3.28-1.06 1.06-3.72-3.72z"></path></svg><div class="Pwlnc f RAyMq"><span class="biGQs _P ZNjnF">Fechado agora</span></div></span></div></div></div></div></div></div></div><div class="NJIJw"><div class="bbfVZ" tabindex="0" role="button"><a class="rmyCe _G B- z _S c Wc wSSLS jWkoZ QHaGY" href="/UserReviewEdit-g303404-d24941804-Reviews-Delicatesse-Belem_State_of_Para.html"><span class="biGQs _P ezezH"><div class="tWvGj"><svg viewBox="0 0 24 24" width="20px" height="20px" class="d Vb egaXP UmNoP" aria-hidden="true"><path fill-rule="evenodd" clip-rule="evenodd" d="M13.788 7.293 6.19 14.868l-.816 3.746 3.724-.839 7.588-7.583zm3.96 1.84-2.898-2.9.556-.554A2.32 2.32 0 0 1 17.02 5c.378 0 .73.104 1.031.315l.01.007.012.008c1.12.757 1.221 2.26.326 3.151zm-7.896 10.01-5.99 1.35q-.032.009-.064.007a.297.297 0 0 1-.29-.36l1.31-6.023 9.529-9.5A3.82 3.82 0 0 1 17.02 3.5c.66 0 1.318.184 1.893.587a3.536 3.536 0 0 1 .546 5.457z"></path></svg>Faça uma avaliação </div></span></a></div></div></div></div></div><div class="qAZoU c"><div class="GMYGA"><div class="mkNRT j"></div><div class="LaMdn j"><div class="aMnVW"><div class="Yzhnw P"><button class="BrOJk u j z _F _S wSSLS tIqAi iNBVo SSqtP" type="button" disabled="" aria-label="1"><span class="kLqdM"><span class="biGQs _P ezezH">1</span></span></button></div></div></div></div></div><div class="qAZoU c"><div class="biGQs _P VImYz ZNjnF"><div class="Ci">Mostrando 1–14 de 14 resultados</div></div></div></div></div></div><div class="IDaDx mvTrV cyIij fluiI SMjpI Iwmxp"><div class="c f e Q3 roAGK tyUdl"><div class="biGQs _P ezezH">O Tripadvisor se esqueceu de algum lugar?</div><a class="rmyCe _G B- z _S c Wc wSSLS w AeLHi QHaGY" href="/CreateListing.html"><span class="biGQs _P ezezH">Incluir um local</span></a></div></div><div class="Pk PY Px PK"><div class="oTksw EFdEd"><div id="slot:300x250-8x8:footer" class="Eufjb j u ChFsW Xd o S"><div id="google_ads_iframe_/5349/ta.ta.br.s/sa.brazil.state_of_para_9__container__" style="border: 0pt none; display: inline-block;"></div></div></div></div></div><div data-automation="restaurant-list-jsonld"><script type="application/ld+json">{"@context":"https:\u002F\u002Fschema.org","@type":"ItemList","itemListOrder":"https:\u002F\u002Fschema.org\u002FItemListOrderAscending","itemListElement":[{"@type":"ListItem","position":1,"item":{"@type":"Restaurant","name":"Camarada Camar\u00E3o Bel\u00E9m - Shopping Boulevard Bel\u00E9m","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d24040497-Reviews-Camarada_Camarao_Belem_Shopping_Boulevard_Belem-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4.9","reviewCount":1700},"priceRange":"$$ - $$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F25\u002F2b\u002F4e\u002Fd2\u002Fcaption.jpg"],"telephone":"+55 91 99250-8599","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66053-000","streetAddress":"Av. Visc. De Souza Franco, 776, Bel\u00E9m, Par\u00E1 66053-000 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.446138,"longitude":-48.48969}}},{"@type":"ListItem","position":2,"item":{"@type":"Restaurant","name":"Restaurante JP Grill","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d14107327-Reviews-Restaurante_JP_Grill-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4.8","reviewCount":12},"priceRange":"$$ - $$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F15\u002Fff\u002F59\u002F63\u002Fphoto0jpg.jpg"],"telephone":"9132051399","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66035-395","streetAddress":"Avenida Comandante Bras de Aguiar 321 T\u00E9rreo, Bel\u00E9m, Par\u00E1 66035-395 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.45498,"longitude":-48.4892}}},{"@type":"ListItem","position":3,"item":{"@type":"Restaurant","name":"Mirakuru Street Food","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d21311161-Reviews-Mirakuru_Street_Food-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"5","reviewCount":4},"priceRange":"$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F1c\u002F1f\u002F3f\u002F28\u002Fe-bem-desse-jeito-mesmo.jpg"],"telephone":"+5591985151515","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66050-520","streetAddress":"Rua Oliveira Belo 434b Umarizal, Bel\u00E9m, Par\u00E1 66050-520 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.440948,"longitude":-48.480328}}},{"@type":"ListItem","position":4,"item":{"@type":"Restaurant","name":"Old House - Blues e Jazz","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d26491778-Reviews-Old_House_Blues_e_Jazz-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4.2","reviewCount":11},"priceRange":"$$$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F2d\u002Ff0\u002F01\u002F6f\u002Fcaption.jpg"],"telephone":"+55 91 98246-6120","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66035-415","streetAddress":"Av. Br\u00E1s de Aguiar, 716 Nazar\u00E9, Bel\u00E9m, Par\u00E1 66035-415 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.454595,"longitude":-48.485096}}},{"@type":"ListItem","position":5,"item":{"@type":"Restaurant","name":"Radisson Hotel Belem","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d7252312-Reviews-Radisson_Hotel_Belem-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"3.9","reviewCount":9},"priceRange":"","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F2a\u002F26\u002F5c\u002F5f\u002Fcaption.jpg"],"telephone":"+55 91 3205-1399","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66035-395","streetAddress":"Avenida Comandante Br\u00E1s de Aguiar, 301 321, Bel\u00E9m, Par\u00E1 66035-395 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.454824,"longitude":-48.489105}}},{"@type":"ListItem","position":6,"item":{"@type":"Restaurant","name":"Na Telha","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d2373296-Reviews-Na_Telha-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4.2","reviewCount":142},"priceRange":"$$ - $$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F11\u002Fb0\u002Fe1\u002F39\u002F20180104-211513-largejpg.jpg"],"telephone":"(91) 3227-0853","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66810-050","streetAddress":"R. Siqueira Mendes, 263, 21 km Cruzeiro, Bel\u00E9m, Par\u00E1 66810-050 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.297331,"longitude":-48.489677}}},{"@type":"ListItem","position":7,"item":{"@type":"Restaurant","name":"Restaurante Casa Mia","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d9558565-Reviews-Restaurante_Casa_Mia-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4","reviewCount":167},"priceRange":"$$ - $$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F22\u002F4f\u002Fa9\u002Fdf\u002Ffachada-da-nossa-casa.jpg"],"telephone":"+55 91 2121 2661","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66035-190","streetAddress":"Travessa Quintino Bocai\u00FAva n\u00BA 1696, Bel\u00E9m, Par\u00E1 66035-190 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.454191,"longitude":-48.48599}}},{"@type":"ListItem","position":8,"item":{"@type":"Restaurant","name":"Esta\u00E7\u00E3o Gourmet","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d2373295-Reviews-Estacao_Gourmet-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4.5","reviewCount":83},"priceRange":"$$ - $$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F06\u002F19\u002F89\u002F34\u002Festacao-gourmet.jpg"],"telephone":"+55 91 3252-1500","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66015-053","streetAddress":"Travessa 1 de Mar\u00E7o, 766, Bel\u00E9m, Par\u00E1 66015-053 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.454065,"longitude":-48.494682}}},{"@type":"ListItem","position":9,"item":{"@type":"Restaurant","name":"Vegas Poker Club","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d10480155-Reviews-Vegas_Poker_Club-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"4.1","reviewCount":38},"priceRange":"$$ - $$$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F11\u002Fa2\u002Fb1\u002Ff9\u002F20171229-183247-largejpg.jpg"],"telephone":"559131990107","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"","streetAddress":"Travessa Boaventura da Silva, 414, Bel\u00E9m, Par\u00E1 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.448867,"longitude":-48.487213}}},{"@type":"ListItem","position":10,"item":{"@type":"Restaurant","name":"Le Bistrot","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d12316068-Reviews-Le_Bistrot-Belem_State_of_Para.html?m=69411","aggregateRating":{"@type":"AggregateRating","ratingValue":"3.7","reviewCount":26},"priceRange":"","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F13\u002F02\u002Fd5\u002Fe9\u002Fnossa-logomarca-desde.jpg"],"telephone":"+55 91 98411-2660","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66020-250","streetAddress":"Rua Dr. Malcher 15 Umarizal, Bel\u00E9m, Par\u00E1 66020-250 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.455697,"longitude":-48.504658}}},{"@type":"ListItem","position":11,"item":{"@type":"Restaurant","name":"Bar e Restaurante La Marine","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d13435710-Reviews-Bar_e_Restaurante_La_Marine-Belem_State_of_Para.html?m=69411","priceRange":"$","image":["https:\u002F\u002Fdynamic-media-cdn.tripadvisor.com\u002Fmedia\u002Fphoto-o\u002F14\u002Fb4\u002F56\u002Fdd\u002Fvoce-pode-almocar-em.jpg"],"telephone":"+559182263367","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66645-455","streetAddress":"Rua Nossa Senhora Aparecida 38 Bairro Castanheira, Bel\u00E9m, Par\u00E1 66645-455 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.408349,"longitude":-48.432224}}},{"@type":"ListItem","position":12,"item":{"@type":"Restaurant","name":"Dalie Mediterran\u00E9","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d27167379-Reviews-Dalie_Mediterrane-Belem_State_of_Para.html?m=69411","priceRange":"","telephone":"","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"","streetAddress":"Tv Quintino Bocaiuva 441 Redito, Bel\u00E9m, Par\u00E1 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.44474,"longitude":-48.49217}}},{"@type":"ListItem","position":13,"item":{"@type":"Restaurant","name":"The Coffee Batista Campos","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d33354040-Reviews-The_Coffee_Batista_Campos-Belem_State_of_Para.html?m=69411","priceRange":"","telephone":"","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"66033-770","streetAddress":"Avenida Serzedelo Corr\u00EAa 401, Bel\u00E9m, Par\u00E1 66033-770 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.46011,"longitude":-48.488503}}},{"@type":"ListItem","position":14,"item":{"@type":"Restaurant","name":"D\u00E9licatesse","description":"","url":"https:\u002F\u002Fwww.tripadvisor.com.br\u002FRestaurant_Review-g303404-d24941804-Reviews-Delicatesse-Belem_State_of_Para.html?m=69411","priceRange":"","telephone":"+55 91 99170-3499","address":{"@type":"PostalAddress","addressCountry":"Brasil","addressLocality":"","addressRegion":"","postalCode":"","streetAddress":"Senador Lemos,792, Bel\u00E9m, Par\u00E1 Brasil"},"geo":{"@type":"GeoCoordinates","latitude":-1.436595,"longitude":-48.487015}}}]}</script></div><!--/$--></main>
"""
# Criando um objeto BeautifulSoup a partir do HTML completo
soup_completo = BeautifulSoup(html_completo_exemplos, 'html.parser')

# Encontrar todos os blocos de restaurante
# A classe 'tbrcR' combinada com outras parece ser um bom seletor para cada "cartão" de restaurante.
restaurant_blocks = soup_completo.find_all('div', class_='tbrcR')

print("Iniciando a extração detalhada de cada restaurante:")
restaurantes_detalhados = [] # Lista para armazenar os dados estruturados

for block in restaurant_blocks:
    # --- Seletores para as informações DENTRO de cada bloco de restaurante ---

    # 1. Nome do Restaurante
    # Pistas: O nome está dentro de uma <a> que tem as classes 'BMQDV' e 'ukgoS'.
    # O texto real do nome está dentro de uma <div> que é filha dessa <a>.
    # Exemplo HTML: <a class="BMQDV ... ukgoS"><div class="biGQs ...">1. Nome do Restaurante</div></a>
    nome_link_tag = block.find('a', class_=lambda c: c and 'BMQDV' in c and 'ukgoS' in c)
    nome = "Nome não encontrado"
    if nome_link_tag:
        # A div específica que contém o texto do nome
        nome_div_tag = nome_link_tag.find('div', class_=lambda c: c and 'biGQs' in c and 'fiohW' in c)
        if nome_div_tag:
            nome = nome_div_tag.get_text(strip=True)

    # 2. Avaliação (e.g., 4,9)
    # Pistas: Encontrar a div com data-automation="bubbleRatingValue" e pegar o texto do span dentro.
    avaliacao_tag = block.find('div', {'data-automation': 'bubbleRatingValue'})
    avaliacao = avaliacao_tag.find('span').get_text(strip=True) if avaliacao_tag else "Avaliação não encontrada"

    # 3. Contagem de Reviews
    # Pistas: Está em uma <div> com o atributo 'data-automation="bubbleReviewCount"'.
    # O texto real da contagem está dentro de um <span> que é filho dessa <div>.
    # Usamos regex para extrair apenas os números.
    review_count_tag = block.find('div', {'data-automation': 'bubbleReviewCount'})
    contagem_reviews = "Contagem de reviews não encontrada"
    if review_count_tag:
        text_content = review_count_tag.get_text(strip=True)
        # Extrair apenas os números da string (ex: "(2.529 avaliações)" -> "2.529")
        match = re.search(r'\d[\d.]*', text_content)
        if match:
            contagem_reviews = match.group(0).replace('.', '') # Remove o ponto para converter para int, se necessário

    # 4. Tipo de Cozinha (ex: Brasileira, Bar)
    # Pistas: É um <span> dentro de um <div> com classes 'ZvrsW N G biqBm'.
    # O texto pode estar diretamente no <span> ou em um <span> aninhado.
    # Tentaremos pegar o primeiro span que não contém um SVG ou um link.
    cuisine_tag = block.find('div', class_=lambda c: c and 'ZvrsW' in c and 'biqBm' in c)
    tipo_cozinha = []
    if cuisine_tag:
        # Encontrar todos os spans que são filhos diretos ou quase diretos e que contêm texto relevante
        spans = cuisine_tag.find_all('span', class_=lambda c: c and 'biGQs' in c and 'pZUbB' in c)
        for span in spans:
            # Excluir spans que contêm SVGs (ícones) ou que parecem ser contadores de reviews
            # e pegar apenas o texto que não é de contagem de reviews ou "Fechado agora" ou "Cardápio"
            if not span.find('svg') and 'avaliações' not in span.get_text() and 'Fechado agora' not in span.get_text() and 'Cardápio' not in span.get_text():
                cuisine_text = span.get_text(strip=True)
                if cuisine_text:
                    tipo_cozinha.append(cuisine_text)
    tipo_cozinha_str = ", ".join(tipo_cozinha) if tipo_cozinha else "Tipo de cozinha não encontrado"

    # 5. Faixa de Preço (ex: $$ - $$$)
    # Pistas: Está em um <span> com as mesmas classes do tipo de cozinha, mas é o que não tem SVG
    # e que corresponde ao padrão de preço.
    faixa_preco = "Faixa de preço não encontrada"
    if cuisine_tag:
        for span in cuisine_tag.find_all('span', class_=lambda c: c and 'biGQs' in c and 'pZUbB' in c):
            text_content = span.get_text(strip=True)
            if re.match(r'\$+', text_content): # Verifica se o texto é um ou mais símbolos de dólar
                faixa_preco = text_content
                break

    # 6. Status de Funcionamento (Aberto/Fechado)
    # Pistas: Está dentro de uma div com classe 'Pwlnc f RAyMq', e o texto real no span dentro.
    status_funcionamento_tag = block.find('div', class_=lambda c: c and 'Pwlnc' in c and 'RAyMq' in c)
    status_funcionamento = status_funcionamento_tag.find('span').get_text(strip=True) if status_funcionamento_tag else "Status de funcionamento não encontrado"

    # 7. URL do Restaurante
    # Pistas: A principal tag <a> que envolve todo o bloco do restaurante.
    # O atributo 'href' contém o caminho relativo. Precisamos concatenar com a base do Tripadvisor.
    url_tag = block.find('a', class_=lambda c: c and 'BMQDV' in c and 'PaRlG' in c)
    url_base = "https://www.tripadvisor.com.br"
    url_restaurante = url_base + url_tag['href'] if url_tag and 'href' in url_tag.attrs else "URL não encontrada"

    # 8. URL da Imagem Principal
    # Pistas: A primeira tag <picture> dentro do carousel, e a tag <img> dentro dela.
    # O atributo 'src' contém a URL da imagem.
    image_tag = block.find('div', class_='IdURT').find('img') if block.find('div', class_='IdURT') else None
    url_imagem = image_tag['src'] if image_tag and 'src' in image_tag.attrs else "URL da imagem não encontrada"

    # Armazenar os dados em um dicionário
    restaurante_info = {
        "Nome": nome,
        "Avaliacao": avaliacao,
        "Contagem_Reviews": contagem_reviews,
        "Tipo_Cozinha": tipo_cozinha_str,
        "Faixa_Preco": faixa_preco,
        "Status_Funcionamento": status_funcionamento,
        "URL_Restaurante": url_restaurante,
        "URL_Imagem_Principal": url_imagem
    }
    restaurantes_detalhados.append(restaurante_info)

# Imprimir os resultados para verificação
for i, restaurante in enumerate(restaurantes_detalhados):
    print(f"\n--- Restaurante {i+1} ---")
    for key, value in restaurante.items():
        print(f"{key}: {value}")